<a href="https://colab.research.google.com/github/agcbls49/THESIS/blob/main/Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INTRODUCTION**
This colab file contains the model training and model comparisons of YOLOv5n, YOLOv8n, YOLOv11n and YOLOv26n using a 80% train, 10% validation and 10% test data split. For more information, you can visit this Github [repository](https://github.com/agcbls49/THESIS).

<br>

---

# **SPECIFICATIONS**

The dataset is split into **80% train, 10% valid and 10% test** and every model is trained using the following default YOLO specifications:
1. 100 Epochs
2. All images resized to 640x640
3. 16 Batches

# **SETUP**

### Check if the correct runtime is set

The runtime should be A100 if the file is run in Google Colab. This is run to ensure that the runtime used is not CPU or a lower powered GPU in Colab.

In [ ]:
!nvidia-smi

### Mounting the Dataset from Google Drive

This step mounts the Google Drive where the dataset is stored and copies the dataset.zip file to Google Colab's storage allowing the dataset to be used for model training.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Copy dataset.zip from Drive to Colab's local storage
!cp "/content/gdrive/MyDrive/2_THESIS_DATASET/dataset2.zip" /content/

### Unzipping the dataset contents
This step simply unzips the content of dataset.zip in Google Colab's storage.

In [3]:
# Unzip the dataset folder contents
!unzip -q /content/dataset2.zip -d /content/dataset

### Installing Ultralytics for YOLO
This step installs the ultralytics library which is needed for training YOLO models.

In [ ]:
# Install ultralytics for YOLO
!pip install ultralytics -q

### Creating a YAML file
This step creates a YAML file which is needed for training YOLO models. It contains the path of the unzipped dataset, where the train, validation and test images are located, and the names of the classes along with their class numbers.

In [ ]:
import yaml
import os

data = {
    'path': '/content/dataset/dataset2',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 24,
    'names': [
        'backpack',
        'book',
        'cup',
        'glasses',
        'headset',
        'key',
        'phone',
        'remote',
        'wallet',
        'water bottle',
        'body spray',
        'card',
        'charger',
        'comb',
        'flashlight',
        'glasses case',
        'medicine',
        'nail clipper',
        'shoe',
        'watch',
        'body scrub',
        'cane',
        'lighter',
        'towel'
    ]
}

yaml_path = '/content/data.yaml'

# write file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("written")

# verify file exists
print("exists:", os.path.exists(yaml_path))

# verify size
print("size:", os.path.getsize(yaml_path))

# print contents
with open(yaml_path, 'r') as f:
    content = f.read()

print("CONTENT:")
print(content)

### **(OPTIONAL)** Deleting Conflicting Cache

**This step is done only if the YOLO commands produce a YAML error otherwise this can be skipped**. <br><br> The code below deletes the cached YAML file ensuring that the code runs the proper yaml file.

In [ ]:
!find /content/dataset/dataset -name "*.cache" -delete

# **RUNNING THE MODELS**

### **YOLO v5 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolov5n.pt epochs=100 imgsz=640 batch=16 name=yolov5n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [7]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolov5n_run/weights/best.pt split=test name=yolov5n_test

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv5n summary (fused): 85 layers, 2,635,592 parameters, 0 gradients, 7.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2147.6±1439.8 MB/s, size: 1813.5 KB)
val: Scanning /content/dataset/dataset2/labels/test... 792 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 1.2Kit/s 0.7s
val: New cache created: /content/dataset/dataset2/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.9it/s 12.8s
                   all        792       1432      0.898      0.828      0.896      0.718
              backpack         39         39       0.95      0.897      0.961      0.835
                  book         50        128      0.816      0.719      0.796      0.625
                   cup         83        169       0.85      0.781      0.812      0.596
               glasses         53         53    

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolov5n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolov5n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolov5n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolov5n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolov5n_test_results.txt")



---



### **YOLO v8 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolov8n.pt epochs=100 imgsz=640 batch=16 name=yolov8n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolov8n_run/weights/best.pt split=test name=yolov8n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2258.0±1117.9 MB/s, size: 1958.9 KB)
val: Scanning /content/dataset/dataset2/labels/test... 792 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 1.1Kit/s 0.7s
val: New cache created: /content/dataset/dataset2/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.8it/s 13.3s
                   all        792       1432      0.907      0.843      0.901      0.736
              backpack         39         39      0.939      0.974      0.987      0.873
                  book         50        128      0.816      0.695      0.776      0.627
                   cup         83        169      0.851      0.812       0.83      0.616
               glasses         53         53      

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolov8n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolov8n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolov8n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolov8n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolov8n_test_results.txt")



---



### **YOLO v11 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11n.pt epochs=100 imgsz=640 batch=16 name=yolo11n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolo11n_run/weights/best.pt split=test name=yolo11n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11n summary (fused): 100 layers, 2,586,832 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3082.3±1938.1 MB/s, size: 2218.4 KB)
val: Scanning /content/dataset/dataset2/labels/test.cache... 792 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 144.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 4.1it/s 12.1s
                   all        792       1432      0.925      0.852      0.903      0.746
              backpack         39         39      0.945      0.974      0.985      0.865
                  book         50        128      0.917      0.719      0.847      0.686
                   cup         83        169       0.88      0.793      0.833      0.637
               glasses         53         53      0.954      0.925      0.971      0.829
               hea

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolo11n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolo11n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolo11n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolo11n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolo11n_test_results.txt")



---



### **YOLO v26 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo26n.pt epochs=100 imgsz=640 batch=16 name=yolo26n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolo26n_run/weights/best.pt split=test name=yolo26n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26n summary (fused): 120 layers, 2,379,516 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1672.5±432.9 MB/s, size: 2676.5 KB)
val: Scanning /content/dataset/dataset2/labels/test... 792 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 1.1Kit/s 0.7s
val: New cache created: /content/dataset/dataset2/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 4.1it/s 12.2s
                   all        792       1432      0.906      0.875      0.913      0.766
              backpack         39         39      0.965      0.974      0.974       0.86
                  book         50        128      0.704      0.695      0.801      0.662
                   cup         83        169      0.854       0.83      0.839      0.656
               glasses         53         53    

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolo26n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolo26n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolo26n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolo26n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolo26n_test_results.txt")



---



**After comparing the results from both data split configurations** *(the other data split is in the [Experiment 1 colab file](https://colab.research.google.com/drive/1ki3rl4tZwEHgTyHqFQJyDXnauPp2_YqN?usp=sharing)),* **the selected model will be further evaluated using 150 training epochs.**

This will be on the [Experiment 3 colab file](https://colab.research.google.com/drive/1AE36BZFs3d6x4c7-419_y96wG4L6LFC7?usp=sharing).